# 01 - Staleness Detector

Identifies tables not read or written in N days using `system.access.table_lineage` and `information_schema.tables`.

**Outputs**: Writes findings to `{catalog}.{control_schema}.scan_results`

In [0]:
# Databricks notebook source
import sys as _sys
_nb = (dbutils.notebook.entry_point.getDbutils().notebook()
       .getContext().notebookPath().get())
_sys.path.insert(0, '/Workspace' + '/'.join(_nb.split('/')[:-2]) + '/src')
from lib.common import (
    require_widget, uc_list_tables, uc_list_schemas,
    tables_to_spark, build_exempt_schemas,
    load_exemptions, is_exempt,
)
dbutils.widgets.text("catalog",         "")
dbutils.widgets.text("control_schema",  "uc_hygiene")
dbutils.widgets.text("target_catalogs", "")
dbutils.widgets.text("staleness_days",  "30")
dbutils.widgets.text("archive_threshold_days", "60")



catalog        = require_widget(dbutils, "catalog")
control_schema = require_widget(dbutils, "control_schema")
target_catalogs = [
    c.strip() for c in require_widget(dbutils, "target_catalogs").split(",") if c.strip()
]
staleness_days  = int(dbutils.widgets.get("staleness_days") or "30")
archive_threshold_days = int(dbutils.widgets.get("archive_threshold_days") or "60")
control_fqn     = f"{catalog}.{control_schema}"

print(f"Control schema:      {control_fqn}")
print(f"Scanning catalogs:   {target_catalogs}")
print(f"Staleness threshold: {staleness_days} days")
import time as _t; _task_start = _t.time()

In [0]:
# Function definitions moved to src/lib/common.py — imported in widget cell above.
from databricks.sdk import WorkspaceClient

_sdk = WorkspaceClient()


print("✅ lib.common loaded; SDK client ready.")


In [0]:
from datetime import date, timedelta
import uuid

scan_id = str(uuid.uuid4())
scan_date = date.today()
cutoff_date = scan_date - timedelta(days=staleness_days)

# Exempted schemas
# Derives from control_schema so forks don't silently scan the wrong schema.
EXEMPT_SCHEMAS = sorted(build_exempt_schemas(control_schema))
exempt_clause = " AND ".join([f"t.table_schema != '{s}'" for s in EXEMPT_SCHEMAS])

print(f"Scan ID: {scan_id}")
print(f"Cutoff date: {cutoff_date}")

In [0]:
# Step 1: Get all tables via UC SDK — no Spark metadata scan
_table_rows = uc_list_tables(_sdk, target_catalogs, control_schema)
_exemptions = load_exemptions(spark, catalog, control_schema)
_table_rows  = [r for r in _table_rows if not is_exempt(r["catalog_name"], r["schema_name"], r["table_name"], _exemptions)]
all_tables   = tables_to_spark(spark, _table_rows).withColumnRenamed("table_catalog", "catalog_name") \
                                            .withColumnRenamed("table_schema", "schema_name")
# created_at and last_altered are populated from the UC SDK — see tables_to_spark()
total_tables = len(_table_rows)
print(f"Total tables to evaluate across {len(target_catalogs)} catalog(s): {total_tables}")


In [0]:
# Step 2: Find last access time per table from lineage
# table_lineage records both reads (source) and writes (target)
last_access_query = f"""
WITH last_read AS (
  SELECT 
    source_table_catalog AS catalog_name,
    source_table_schema AS schema_name,
    source_table_name AS table_name,
    MAX(event_time) AS last_read_at
  FROM system.access.table_lineage
  WHERE source_table_catalog IN ({','.join([f"'{c}'" for c in target_catalogs])})
    AND event_time >= '{cutoff_date}'
  GROUP BY 1, 2, 3
),
last_write AS (
  SELECT 
    target_table_catalog AS catalog_name,
    target_table_schema AS schema_name,
    target_table_name AS table_name,
    MAX(event_time) AS last_write_at
  FROM system.access.table_lineage
  WHERE target_table_catalog IN ({','.join([f"'{c}'" for c in target_catalogs])})
    AND event_time >= '{cutoff_date}'
  GROUP BY 1, 2, 3
)
SELECT 
  COALESCE(r.catalog_name, w.catalog_name) AS catalog_name,
  COALESCE(r.schema_name, w.schema_name) AS schema_name,
  COALESCE(r.table_name, w.table_name) AS table_name,
  r.last_read_at,
  w.last_write_at,
  GREATEST(COALESCE(r.last_read_at, w.last_write_at), COALESCE(w.last_write_at, r.last_read_at)) AS last_activity
FROM last_read r
FULL OUTER JOIN last_write w
  ON r.catalog_name = w.catalog_name
  AND r.schema_name = w.schema_name
  AND r.table_name = w.table_name
"""

active_tables = spark.sql(last_access_query)
print(f"Tables with recent activity: {active_tables.count()}")

In [0]:
# Step 3: Identify stale tables (in all_tables but NOT in active_tables)
all_tables.createOrReplaceTempView("all_tables")
active_tables.createOrReplaceTempView("active_tables")

stale_tables = spark.sql(f"""
SELECT 
  a.catalog_name,
  a.schema_name,
  a.table_name,
  a.created_at,
  a.last_altered,
  act.last_activity,
  -- Use lineage-based last_activity as ground truth. Fall back to created_at
  -- (NOT last_altered — metadata changes like tag edits update last_altered
  -- without real data access, making never-accessed tables appear fresh).
  DATEDIFF(CURRENT_DATE(), COALESCE(act.last_activity, a.created_at)) AS days_since_activity
FROM all_tables a
LEFT JOIN active_tables act
  ON a.catalog_name = act.catalog_name
  AND a.schema_name = act.schema_name
  AND a.table_name = act.table_name
WHERE act.last_activity IS NULL
  AND a.created_at < '{cutoff_date}'
ORDER BY days_since_activity DESC
""")

stale_count = stale_tables.count()
print(f"🚨 Stale tables found: {stale_count}")
if stale_count > 0:
    stale_tables.show(20, truncate=False)

In [0]:
# Step 4: Resolve owner from existing tags — SQL union across all target catalogs
_tag_parts = " UNION ALL ".join([
    f"SELECT catalog_name, schema_name, table_name, tag_value AS owner_email "
    f"FROM {tc}.information_schema.table_tags WHERE tag_name = 'owner'"
    for tc in target_catalogs
])
owner_tags = spark.sql(_tag_parts)
owner_tags.createOrReplaceTempView("owner_tags")

# Step 5: Write stale findings to control table
stale_tables.createOrReplaceTempView("stale_findings")

spark.sql(f"""
INSERT INTO {catalog}.{control_schema}.scan_results
SELECT
  '{scan_id}' AS scan_id,
  CURRENT_DATE() AS scan_date,
  'staleness' AS scan_type,
  'table'    AS asset_type,
  sf.catalog_name, sf.schema_name, sf.table_name,
  NULL AS column_name,
  CASE WHEN sf.days_since_activity > {archive_threshold_days} THEN 'stale_critical' ELSE 'stale_warning' END AS finding_type,
  CASE WHEN sf.days_since_activity > {archive_threshold_days} THEN 'critical' ELSE 'warning' END AS finding_severity,
  CONCAT('No activity for ', sf.days_since_activity, ' days. Last activity: ',
         COALESCE(CAST(sf.last_activity AS STRING), CAST(sf.last_altered AS STRING), 'unknown')) AS finding_detail,
  CASE WHEN sf.days_since_activity > {archive_threshold_days}
       THEN 'Archive or confirm still needed within 7 days'
       ELSE 'Confirm table is still needed or mark for deprecation'
  END AS recommended_action,
  ot.owner_email,
  NULL AS resolved_at,
  NULL AS resolved_by
FROM stale_findings sf
LEFT JOIN owner_tags ot
  ON sf.catalog_name = ot.catalog_name
  AND sf.schema_name = ot.schema_name
  AND sf.table_name = ot.table_name
""")
print(f"✅ Wrote {stale_count} staleness findings to {catalog}.{control_schema}.scan_results")


In [0]:
_duration = int(_t.time() - _task_start)

print(f"""
{'='*52}
  STALENESS SCAN COMPLETE
{'='*52}
  Scan ID:        {scan_id}
  Date:           {scan_date}
  Tables scanned: {total_tables}
  Stale found:    {stale_count}
  Results → {control_fqn}.scan_results
{'='*52}
""")

# Write execution record
try:
    spark.sql(f"""
    INSERT INTO {control_fqn}.job_run_history VALUES (
      DATE('{scan_date}'),
      'uc_hygiene_daily_governance',
      'p2_staleness',
      'p2_detection',
      'success',
      {total_tables},
      {stale_count},
      {stale_count},
      {_duration},
      'staleness_days={staleness_days} catalogs={len(target_catalogs)}',
      CURRENT_TIMESTAMP()
    )
    """)
except Exception as _e:
    print(f"Warning: could not write to job_run_history: {_e}")